In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "sumy", "bert-score", "sacrebleu"], check=True)

import os
import math
import numpy as np
import pandas as pd
import torch
import chardet
import evaluate
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # для Colab без GUI

# sumy — для экстрактивных методов
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer   # PageRank на графе предложений
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer as SumyStemmer
from sumy.utils import get_stop_words

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ruT5_science_sum_gazeta"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "overfit_checkpoints")
FINAL_MODEL_DIR = os.path.join(PROJECT_DIR, "overfit_final_model")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model_checkpoint = "IlyaGusev/rut5_base_sum_gazeta"

max_input_length = 600
max_target_length = 200

device = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("\n" + "="*50)
print("Загрузка IIS датасета...")
os.system("rm -rf /content/summarization-dataset")
os.system("git clone https://github.com/iis-research-team/summarization-dataset.git /content/summarization-dataset")

texts_iis, abstracts_iis = [], []
data_path = "/content/summarization-dataset/dataset"

for domain in os.listdir(data_path):
    domain_path = os.path.join(data_path, domain)
    if os.path.isdir(domain_path):
        for paper_folder in os.listdir(domain_path):
            paper_path = os.path.join(domain_path, paper_folder)
            text_file = os.path.join(paper_path, "text.txt")
            abstract_file = os.path.join(paper_path, "abstract.txt")

            if os.path.exists(text_file) and os.path.exists(abstract_file):
                with open(text_file, "r", encoding="utf-8") as f:
                    text = f.read().strip()
                with open(abstract_file, "r", encoding="utf-8") as f:
                    abstract = f.read().strip()

                if text and abstract:
                    texts_iis.append(text)
                    abstracts_iis.append(abstract)

print(f"IIS: загружено {len(texts_iis)} пар")

In [ ]:
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
        result = chardet.detect(raw_data)
        return result['encoding'] or 'utf-8'

def read_file_with_encoding(file_path):
    encoding = detect_encoding(file_path)
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            return f.read()
    except Exception:
        for enc in ['utf-8', 'windows-1251', 'cp1251', 'koi8-r', 'iso-8859-5']:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    return f.read()
            except Exception:
                continue
        raise UnicodeDecodeError(f"Не удалось прочитать файл {file_path}")

print("\n" + "="*50)
print("Загрузка RuSciTextSum...")
os.system("rm -rf /content/RuSciTextSum")
os.system("git clone https://github.com/Svetych/RuSciTextSum.git /content/RuSciTextSum")

# --- RuSciText ---
texts_rusci, abstracts_rusci = [], []
rusci_path = "/content/RuSciTextSum/dataset/ruscitext"

if os.path.exists(rusci_path):
    for filename in os.listdir(rusci_path):
        file_path = os.path.join(rusci_path, filename)
        if os.path.isfile(file_path):
            try:
                content = read_file_with_encoding(file_path)
                lines = content.strip().split('\n\n')
                if len(lines) >= 2:
                    text = '\n\n'.join(lines[:-1]).strip()
                    abstract = lines[-1].strip()
                    if text and abstract:
                        texts_rusci.append(text)
                        abstracts_rusci.append(abstract)
            except Exception as e:
                print(f"  Ошибка при чтении {filename}: {e}")
else:
    print("⚠️ Путь ruscitext не найден:", rusci_path)

print(f"RuSciText: загружено {len(texts_rusci)} пар")

# --- RuArchive ---
texts_archive, abstracts_archive = [], []
archive_path = "/content/RuSciTextSum/dataset/ruarchive"

if os.path.exists(archive_path):
    for filename in os.listdir(archive_path):
        file_path = os.path.join(archive_path, filename)
        if os.path.isfile(file_path):
            try:
                content = read_file_with_encoding(file_path)
                lines = content.strip().split('\n\n')
                if len(lines) >= 2:
                    text = lines[0].strip()
                    abstract = lines[1].strip()
                    if text and abstract:
                        texts_archive.append(text)
                        abstracts_archive.append(abstract)
            except Exception as e:
                print(f"  Ошибка при чтении {filename}: {e}")
else:
    print("⚠️ Путь ruarchive не найден:", archive_path)

print(f"RuArchive: загружено {len(texts_archive)} пар")
